# transformer-inference-lab — MHA training run

Trains the MHA baseline checkpoint (`n_kv_head=8`) for the memory–latency–quality comparison against GQA and MQA. Run as **Save & Run All (Commit)**, not interactive — this takes ~2.8h on a T4.

**Before running:** attach the `transformer-inference-lab-data` dataset as an Input (sidebar → Add Input), and confirm `configs/mha.yaml` has `max_iters: 5000` on GitHub.

## 1. GPU check + clone repo

In [ ]:
!nvidia-smi
!git clone https://github.com/modestesavadogo/transformer-inference-lab.git
%cd /kaggle/working/transformer-inference-lab
!git log --oneline -5

## 2. Install dependencies

In [ ]:
!pip install -r requirements.txt --quiet

## 3. Link the dataset
Requires `transformer-inference-lab-data` already attached via the notebook's Input sidebar.

In [ ]:
import os
os.listdir("/kaggle/input/datasets/modestesavadogomaths/transformer-inference-lab-data")

In [ ]:
!mkdir -p data
!ln -s /kaggle/input/datasets/modestesavadogomaths/transformer-inference-lab-data/train.bin data/train.bin
!ln -s /kaggle/input/datasets/modestesavadogomaths/transformer-inference-lab-data/val.bin data/val.bin
!ls -la data/

## 4. Verify data before committing to a multi-hour run

In [ ]:
import numpy as np

train_data = np.memmap('data/train.bin', dtype=np.uint16, mode='r')
val_data = np.memmap('data/val.bin', dtype=np.uint16, mode='r')
print(f"train: {len(train_data):,} tokens")
print(f"val:   {len(val_data):,} tokens")

assert len(train_data) > 40_000_000, "train.bin looks too small — check dataset link"
assert len(val_data) > 10_000, "val.bin looks too small — check dataset link"
print("data check passed")

## 5. Confirm config is set for the real run
Read the output before running the training cell. Confirm `max_iters: 5000`, `batch_size: 8`, `grad_accum_steps: 8`. If it still shows `max_iters: 30`, stop — pull the fix from GitHub first.

In [ ]:
!cat configs/mha.yaml

## 6. Run tests — cheap insurance before a multi-hour job

In [ ]:
!python -m pytest tests/ -q

## 7. Train MHA (the ~2.8h cell)
Runs to completion in the background under Save & Run All, even if the tab is closed.

In [ ]:
!python train.py --config configs/mha.yaml --device cuda --eval-interval 250 --log-interval 50

## 8. Confirm the checkpoint is valid

In [ ]:
import torch

ckpt = torch.load("results/checkpoints/mha.pt", map_location="cuda")
print("iter:", ckpt["iter"])
print("val_loss:", ckpt.get("val_loss"))
print("config:", ckpt["config"])
print("num tensors in state dict:", len(ckpt["model_state_dict"]))